In [0]:
dbutils.widgets.dropdown("subfolder", "orders_cdc",["orders_cdc", "order_items_cdc", "customers_cdc"],"CDC Source",)

In [0]:
dbutils.widgets.dropdown("catalog", "dev", ["dev", "prod"])

In [0]:
subfolder = dbutils.widgets.get("subfolder")
catalog = dbutils.widgets.get("catalog")
print(f"Selected source: {subfolder} and catalog: {catalog}")


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType, BooleanType, LongType, DoubleType)

In [0]:
target_table= f"{catalog}.os_stepright.bronze_{subfolder}"
print(target_table)

In [0]:
checkpoint_loc = f"/Volumes/{catalog}/os_stepright/checkpoints/bronze_{subfolder}"
print(checkpoint_loc)

In [0]:
orders_cdc_struct_column_schema = StructType(
    [
        StructField("order_id", StringType()),
        StructField("customer_id", StringType()),
        StructField("order_status", StringType()),
        StructField("order_date", StringType()),
        StructField("updated_at", StringType()),
        StructField("shipping_address_id", StringType()),
        StructField("shipping_city", StringType()),
        StructField("shipping_state", StringType()),
        StructField("shipping_country", StringType()),
        StructField("payment_method", StringType()),
        StructField("discount_code", StringType()),
        StructField("discount_amount", DoubleType()),
        StructField("total_amount", DoubleType()),
    ]
)

In [0]:
order_items_cdc_struct_column_schema = StructType([
    StructField("order_item_id", StringType()),
    StructField("order_id", StringType()),
    StructField("product_id", StringType()),
    StructField("sku", StringType()),
    StructField("quantity", LongType()),
    StructField("unit_price", DoubleType()),
    StructField("line_total", DoubleType()),
    StructField("return_requested", BooleanType()),
    StructField("return_reason", StringType()),
])

In [0]:
customers_cdc_struct_column_schema = StructType([
    StructField("customer_id", StringType()),
    StructField("email", StringType()),
    StructField("first_name", StringType()),
    StructField("last_name", StringType()),
    StructField("phone", StringType()),
    StructField("date_of_birth", StringType()),
    StructField("gender", StringType()),
    StructField("registration_date", StringType()),
    StructField("loyalty_tier", StringType()),
    StructField("address_line1", StringType()),
    StructField("address_line2", StringType()),
    StructField("city", StringType()),
    StructField("state", StringType()),
    StructField("zip_code", StringType()),
    StructField("country", StringType()),
    StructField("is_active", BooleanType()),
    StructField("updated_at", StringType()),
])

In [0]:
ROW_SCHEMA_MAP = {
    "orders_cdc": orders_cdc_struct_column_schema,
    "order_items_cdc": order_items_cdc_struct_column_schema,
    "customers_cdc": customers_cdc_struct_column_schema,
}

row_schema = ROW_SCHEMA_MAP[subfolder]   # picked automatically based on the dropdown you already selected
print(row_schema)

In [0]:
def envelope_schema(cdc_struct_column_schema: StructType):

    envelope_cdc_schema = StructType(
    [
        StructField("op", StringType()),
        StructField("ts_ms", LongType()),
        StructField("before", cdc_struct_column_schema),
        StructField("after", cdc_struct_column_schema),
    ]
    )

    return envelope_cdc_schema

In [0]:
def read_cdc_source(catalog,folder_path, _schema):
    
    df_cdc = (
        spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .schema(envelope_schema(_schema))
            .load(f"/Volumes/{catalog}/stepright/landing/{folder_path}/")
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_source_file", F.col("_metadata.file_path"))
    )
    return df_cdc

In [0]:
def write_cdc_target(read_df,checkpoint_loc):
    (
        read_df.writeStream
            .format("delta")
            .option("checkpointLocation", checkpoint_loc)
            .outputMode("append")
            .trigger(availableNow=True)
            .toTable(target_table)
    )

In [0]:
orders_df = read_cdc_source(catalog, subfolder, row_schema)
write_cdc_target(orders_df, checkpoint_loc)

In [0]:
%sql
select * from dev.os_stepright.bronze_orders_cdc

In [0]:
%sql
select * from dev.os_stepright.bronze_categories

In [0]:
%sql
desc history dev.os_stepright.bronze_categories

In [0]:
%sql
select 'bronze_products' as table_name, count(*) as row_count from dev.os_stepright.bronze_products
union all
select 'bronze_categories', count(*) from dev.os_stepright.bronze_categories
union all
select 'bronze_inventory', count(*) from dev.os_stepright.bronze_inventory
union all
select 'bronze_clickstream', count(*) from dev.os_stepright.bronze_clickstream
union all
select 'bronze_orders_cdc', count(*) from dev.os_stepright.bronze_orders_cdc
union all
select 'bronze_order_items_cdc', count(*) from dev.os_stepright.bronze_order_items_cdc
union all
select 'bronze_customers_cdc', count(*) from dev.os_stepright.bronze_customers_cdc
union all
select 'products_valid', count(*) from dev.os_stepright.bronze_products_valid
union all
select 'products_quarantined', count(*) from dev.os_stepright.bronze_products_quarantined
union all
select 'categories_valid', count(*) from dev.os_stepright.bronze_categories_valid
union all
select 'categories_quarantined', count(*) from dev.os_stepright.bronze_categories_quarantined
union all
select 'inventory_valid', count(*) from dev.os_stepright.bronze_inventory_valid
union all
select 'inventory_quarantined', count(*) from dev.os_stepright.bronze_inventory_quarantined
union all
select 'clickstream_valid', count(*) from dev.os_stepright.bronze_clickstream_valid
union all
select 'clickstream_quarantined', count(*) from dev.os_stepright.bronze_clickstream_quarantined
union all
select 'orders_valid', count(*) from dev.os_stepright.bronze_orders_valid
union all
select 'orders_quarantined', count(*) from dev.os_stepright.bronze_orders_quarantined
union all
select 'order_items_valid', count(*) from dev.os_stepright.bronze_order_items_valid
union all
select 'order_items_quarantined', count(*) from dev.os_stepright.bronze_order_items_quarantined
union all
select 'customers_valid', count(*) from dev.os_stepright.bronze_customers_valid
union all
select 'customers_quarantined', count(*) from dev.os_stepright.bronze_customers_quarantined